# Regularization: Fighting Overfitting

Reach for this when you need: 
- Reference for standard regularization techniques in PyTorch.
- To implement dropout and label smoothing.
- Understanding weight decay (L2) vs manual normalization.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Dropout

**nn.Dropout(p)**
Randomly zeroes some elements of the input tensor with probability `p` during training.

✅ **Use when**: Fully connected (Dense) layers and Transformer attention blocks.
❌ **Don't use when**: Convolutional layers (where features are spatially correlated; use `nn.Dropout2d`).

In [ ]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(10, 10)
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x):
        # Dropout automatically turns off during model.eval()
        return self.dropout(self.fc(x))

## 2. Weight Decay (L2 Regularization)

| Method | Where to Apply | Usage |
| :--- | :--- | :--- |
| `optimizer` param | `optim.AdamW(..., weight_decay=1e-5)` | Industry standard for all weights |
| Manual L1 | Inside training loop: `loss += lambda * torch.norm(weights, 1)` | For sparse feature selection |
| Layer Norm | `nn.LayerNorm` | Stabilization in Transformers |

In [ ]:
# Preferred: Optimizer-integrated weight decay
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)

## 3. Label Smoothing

Instead of target being `[1, 0, 0]`, use `[0.9, 0.05, 0.05]`. Prevents the model from becoming over-confident.

✅ **Use when**: Training large classifiers on noise-prone or highly imbalanced data.
❌ **Don't use when**: The ground-truth is absolutely certain and small differences matter.

In [ ]:
# Integrated into CrossEntropyLoss in PyTorch 2.x
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

### Common Pitfalls
- **Evaluation mode**: Dropout and BatchNorm use running statistics and binary masks that MUST be toggled via `model.eval()`. Failure to do so ruins inference.
- **Over-regularization**: Too much dropout or weight decay can prevent the model from learning anything (underfitting).
- **L1 vs L2**: L2 (Weight Decay) pushes all weights toward zero; L1 pushes MANY weights to EXACTLY zero (sparsity).

### Key Takeaways
- Regularization is most effective when the gap between training and validation error is large.
- `AdamW` is preferred over `Adam` because it decouples weight decay from the adaptive learning rate update.
- `Label Smoothing` is a cheap performance boost for modern deep classifiers.